# Part 1: Automated Essay Scoring (AES)
## Comparing Algorithmic and Generative Approaches to L2 Writing Assessment

**Course:** Computer-Assisted Language Learning with Python

This notebook is the final official submission version for Part 1. It uses:
- the official ELLIPSE train/test split
- feature-based AES with linear regression
- LLM-based AES with 12 OpenRouter conditions
- the official holistic rubric from `ELL_Rubrics.docx`

### Notebook roadmap

| Section | Topic | What this version does |
|---------|-------|------------------------|
| 0 | Setup | Install/import packages and load data |
| 1 | Understanding the data | Inspect the official train/test files |
| 2 | AES concepts and baselines | Evaluate simple baselines on the official test set |
| 3 | Task A - Linear Regression | Train on official train and evaluate on official test |
| 4 | Task B - LLM Scoring | Compare 12 LLM conditions on a stratified official-test subset |
| 5 | Save results | Export the final CSV and JSON files for submission |

### What to do

- Run Sections 0-3 to reproduce the feature-based AES pipeline.
- In Section 4, enter your OpenRouter API key and run the 12 official LLM conditions.
- In Section 5, save the final `part1_predictions.csv` and `part1_config.json`.


## Section 0: Setup
Run the following cells to install packages, import libraries, and download the data.

In [ ]:
# Install required packages (this may take a minute)
!pip install pandas scikit-learn scipy matplotlib openai -q
print('All packages installed!')

In [ ]:
# Import all the libraries we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import json
import time
import re
from getpass import getpass

print('All libraries imported!')

In [ ]:
# Load the official ELLIPSE train/test files
TRAIN_URL = 'https://raw.githubusercontent.com/scrosseye/ELLIPSE-Corpus/main/ELLIPSE_Final_github_train.csv'
TEST_URL = 'https://raw.githubusercontent.com/scrosseye/ELLIPSE-Corpus/main/ELLIPSE_Final_github_test.csv'

TRAIN_CANDIDATES = [
    Path('ELLIPSE_Final_github_train.csv'),
    Path('data/ELLIPSE_Final_github_train.csv'),
    Path('D:/ELLIPSE_Final_github_train.csv'),
]
TEST_CANDIDATES = [
    Path('ELLIPSE_Final_github_test.csv'),
    Path('data/ELLIPSE_Final_github_test.csv'),
    Path('D:/ELLIPSE_Final_github_test.csv'),
]

def load_ellipse_csv(local_candidates, remote_url, split_name):
    for candidate in local_candidates:
        if candidate.exists():
            data = pd.read_csv(candidate)
            print(f'Loaded {split_name} data from local file: {candidate}')
            return data, str(candidate)

    data = pd.read_csv(remote_url)
    print(f'Loaded {split_name} data from GitHub: {remote_url}')
    return data, remote_url

train_df, TRAIN_SOURCE = load_ellipse_csv(TRAIN_CANDIDATES, TRAIN_URL, 'train')
test_df, TEST_SOURCE = load_ellipse_csv(TEST_CANDIDATES, TEST_URL, 'test')

ID_CANDIDATES = ['essay_id', 'text_id_kaggle', 'text_id', 'id']
ESSAY_ID_COL = next((c for c in ID_CANDIDATES if c in train_df.columns and c in test_df.columns), None)

if ESSAY_ID_COL is None:
    ESSAY_ID_COL = 'essay_id'
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[ESSAY_ID_COL] = [f'train_{i:05d}' for i in range(len(train_df))]
    test_df[ESSAY_ID_COL] = [f'test_{i:05d}' for i in range(len(test_df))]
    print('No shared essay ID column found. Generated a stable essay_id column for both splits.')
else:
    print(f'Using "{ESSAY_ID_COL}" as the essay ID column.')

# Keep Section 1 focused on the official training set.
df = train_df.copy()

print(f'Training data shape: {train_df.shape}')
print(f'Official test data shape: {test_df.shape}')

---
## Section 1: Understanding Your Data — CSV Files & Pandas

### What is a CSV file?

**CSV** stands for **Comma-Separated Values**. It is a plain text file where:
- Each **row** is one line of text (like a row in Excel).
- Each **column** is separated by a comma (like a column in Excel).

Example of what a CSV looks like inside:
```
name,age,score
Alice,22,3.5
Bob,25,4.0
```

### What is Pandas?

**Pandas** is a Python library that lets you work with tabular data (like spreadsheets). When you load a CSV file with Pandas, it creates a **DataFrame** — think of it as a smart, programmable Excel spreadsheet.

Key vocabulary:
- **DataFrame**: the entire table (all rows and columns)
- **Row**: one data entry (in our case, one essay)
- **Column**: one attribute (e.g., the essay text, the score, word count)
- **Index**: the row number (starting from 0)

In [ ]:
# How big are the official train/test files?
print('Training data (used for exploration below):')
print(f'  Number of essays (rows): {train_df.shape[0]}')
print(f'  Number of columns:       {train_df.shape[1]}')
print('\nOfficial test data:')
print(f'  Number of essays (rows): {test_df.shape[0]}')
print(f'  Number of columns:       {test_df.shape[1]}')

In [ ]:
# What are all the column names?
print('All columns in the dataset:\n')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col}')

In [ ]:
# Identify which column contains the essay text
text_cols = df.select_dtypes(include=['object']).columns.tolist()
print('Text/string columns found:', text_cols)

# Auto-detect the text column
if 'full_text' in df.columns:
    TEXT_COL = 'full_text'
elif 'text' in df.columns:
    TEXT_COL = 'text'
elif len(text_cols) > 0:
    # Use the column with the longest average string length
    avg_lens = {c: df[c].astype(str).str.len().mean() for c in text_cols}
    TEXT_COL = max(avg_lens, key=avg_lens.get)
else:
    TEXT_COL = None

print(f'\n✅ Using "{TEXT_COL}" as the essay text column.')
if TEXT_COL is None:
    print('⚠️ No text column found! LLM scoring (Section 4) will not work.')
    print('   Please check the CSV columns above and set TEXT_COL manually.')

In [ ]:
# Look at the first 5 rows (Pandas shows a nice table)
df.head()

In [ ]:
# Formal schema audit: compare columns and dtypes across official train/test
train_cols = set(train_df.columns)
test_cols = set(test_df.columns)
common_cols = sorted(train_cols & test_cols)
train_only_cols = sorted(train_cols - test_cols)
test_only_cols = sorted(test_cols - train_cols)

print('Schema audit summary')
print('-' * 60)
print(f'Common columns:      {len(common_cols)}')
print(f'Train-only columns:  {len(train_only_cols)} -> {train_only_cols if train_only_cols else "None"}')
print(f'Test-only columns:   {len(test_only_cols)} -> {test_only_cols if test_only_cols else "None"}')
print()

dtype_comparison = pd.DataFrame({
    'train_dtype': train_df[common_cols].dtypes.astype(str),
    'test_dtype': test_df[common_cols].dtypes.astype(str),
})
dtype_comparison['dtype_match'] = dtype_comparison['train_dtype'] == dtype_comparison['test_dtype']

print(f'Dtype mismatches across shared columns: {(~dtype_comparison["dtype_match"]).sum()}')
dtype_comparison


In [ ]:
# Missing-value audit for key columns and candidate AES features
score_cols = ['Overall', 'Cohesion', 'Syntax', 'Vocabulary', 'Phraseology', 'Grammar', 'Conventions']
candidate_feature_cols = [
    'num_words', 'num_sent', 'num_para', 'num_word_div_para',
    'MTLD', 'TTR', 'Type', 'Token'
]

audit_cols = []
for col in [ESSAY_ID_COL, TEXT_COL] + score_cols + candidate_feature_cols:
    if col is not None and col in train_df.columns and col in test_df.columns and col not in audit_cols:
        audit_cols.append(col)

missing_summary = pd.DataFrame({
    'train_missing': train_df[audit_cols].isna().sum(),
    'train_missing_pct': (train_df[audit_cols].isna().mean() * 100).round(2),
    'test_missing': test_df[audit_cols].isna().sum(),
    'test_missing_pct': (test_df[audit_cols].isna().mean() * 100).round(2),
}).sort_values(['train_missing', 'test_missing'], ascending=False)

print('Missing-value audit for key columns')
print('-' * 60)
print(f'Columns checked: {len(audit_cols)}')
print(f'Text column:     {TEXT_COL}')
print(f'Essay ID column: {ESSAY_ID_COL}')
print()
missing_summary


### Understanding the score columns

The ELLIPSE corpus includes **human ratings** for each essay on several dimensions:

| Column | What it measures |
|--------|------------------|
| **Overall** | Holistic language proficiency (this is our **main target**) |
| Cohesion | How well ideas are connected |
| Syntax | Sentence structure quality |
| Vocabulary | Range and accuracy of word choice |
| Phraseology | Use of phrases and expressions |
| Grammar | Grammatical accuracy |
| Conventions | Spelling, punctuation, capitalization |

All scores range from **1.0** (lowest) to **5.0** (highest), in **0.5 increments**.

The dataset also includes **pre-computed linguistic features** like:
- `num_words` — number of words
- `num_sent` — number of sentences  
- `TTR` — Type-Token Ratio (vocabulary diversity)
- `MTLD` — Measure of Textual Lexical Diversity
- `num_word_div_para` — average words per paragraph

In [ ]:
# Read a single essay and its scores
idx = 0  # Try changing this number to read different essays

print('=' * 60)
print(f'ESSAY #{idx}')
print('=' * 60)

if TEXT_COL:
    essay_text = str(df[TEXT_COL].iloc[idx])
    # Show first 500 characters
    print(essay_text[:500])
    if len(essay_text) > 500:
        print('... [truncated]')
    print()

print('Scores:')
score_cols = ['Overall', 'Cohesion', 'Syntax', 'Vocabulary', 'Phraseology', 'Grammar', 'Conventions']
for col in score_cols:
    if col in df.columns:
        print(f'  {col:<15} {df[col].iloc[idx]}')

print(f'\nWord count: {df["num_words"].iloc[idx]}')
print(f'Sentences:  {df["num_sent"].iloc[idx]}')

In [ ]:
# Compare score and text-length distributions across official train/test
train_text_len = train_df[TEXT_COL].astype(str).str.len()
test_text_len = test_df[TEXT_COL].astype(str).str.len()

split_summary = pd.DataFrame([
    {
        'split': 'official_train',
        'n_essays': len(train_df),
        'overall_mean': train_df['Overall'].mean(),
        'overall_std': train_df['Overall'].std(),
        'overall_min': train_df['Overall'].min(),
        'overall_max': train_df['Overall'].max(),
        'text_char_mean': train_text_len.mean(),
        'text_char_median': train_text_len.median(),
    },
    {
        'split': 'official_test',
        'n_essays': len(test_df),
        'overall_mean': test_df['Overall'].mean(),
        'overall_std': test_df['Overall'].std(),
        'overall_min': test_df['Overall'].min(),
        'overall_max': test_df['Overall'].max(),
        'text_char_mean': test_text_len.mean(),
        'text_char_median': test_text_len.median(),
    },
]).round(2)

print('Official split comparison')
print('-' * 60)
split_summary

plt.figure(figsize=(9, 4))
plt.hist(train_df['Overall'], bins=np.arange(0.75, 5.76, 0.25), alpha=0.55, label='Official train', color='steelblue', edgecolor='black')
plt.hist(test_df['Overall'], bins=np.arange(0.75, 5.76, 0.25), alpha=0.55, label='Official test', color='darkorange', edgecolor='black')
plt.xlabel('Overall Holistic Score', fontsize=12)
plt.ylabel('Number of Essays', fontsize=12)
plt.title('Overall Score Distribution: Official Train vs Official Test', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## Section 2: AES Concepts, Evaluation Metrics & Baselines

### What is Automated Essay Scoring (AES)?

**Automated Essay Scoring** is the task of using a computer to predict the score that a human rater would give to an essay. In our case, we want to predict the **Overall holistic proficiency score** (1.0–5.0).

There are two main approaches:
1. **Feature-based**: Extract measurable properties of the text (word count, vocabulary diversity, etc.) and use a statistical model (like linear regression) to predict the score.
2. **LLM-based**: Ask a Large Language Model to read the essay and output a score directly.

We will try **both** and compare them.

### How do we measure "how good" a scoring system is?

We compare our **predicted scores** to the **actual human scores** using two metrics.

---

**1. Pearson Correlation (r)**

Pearson r measures whether your predictions *move in the same direction* as the human scores. When the human gives a high score, does your system also give a high score?

- r = 1.0 → perfect match in direction
- r = 0.0 → no relationship at all
- For AES, r > 0.5 is decent; r > 0.7 is good.

**Toy Example:**

Suppose we have 5 essays:

| Essay | Human score (\(x\)) | Predicted score (\(y\)) |
|-------|---------------------|-------------------------|
| A     | 2.0                 | 2.5                     |
| B     | 3.0                 | 2.8                     |
| C     | 4.0                 | 3.5                     |
| D     | 1.0                 | 1.5                     |
| E     | 5.0                 | 4.7                     |

**Step 1 — Get the means:**

$$
\bar{x} = (2.0 + 3.0 + 4.0 + 1.0 + 5.0) \div 5 = 3.0
$$

$$
\bar{y} = (2.5 + 2.8 + 3.5 + 1.5 + 4.7) \div 5 = 3.0
$$

**Step 2 — For each essay, subtract the mean:**

| Essay | \(x_i - \bar{x}\) | \(y_i - \bar{y}\) | Product |
|-------|--------------------|--------------------|---------|
| A     | −1.0               | −0.5               | 0.5     |
| B     | 0.0                | −0.2               | 0.0     |
| C     | 1.0                | 0.5                | 0.5     |
| D     | −2.0               | −1.5               | 3.0     |
| E     | 2.0                | 1.7                | 3.4     |

**Step 3 — Plug in:**

$$
r = \frac{0.5 + 0.0 + 0.5 + 3.0 + 3.4}{\sqrt{1.0 + 0.0 + 1.0 + 4.0 + 4.0} \;\times\; \sqrt{0.25 + 0.04 + 0.25 + 2.25 + 2.89}}
$$

$$
= \frac{7.4}{\sqrt{10.0} \times \sqrt{5.68}} = \frac{7.4}{3.162 \times 2.383} \approx 0.982
$$

This is very close to 1.0 — our predictions track the human scores almost perfectly.

> **In Python, you don't need to do this by hand:**
> ```python
> from scipy.stats import pearsonr
> r, p = pearsonr([2.0, 3.0, 4.0, 1.0, 5.0],
>                 [2.5, 2.8, 3.5, 1.5, 4.7])
> print(f"Pearson r = {r:.3f}")  # 0.982
> ```

---

**2. Mean Squared Error (MSE)**

While Pearson r tells us whether predictions go in the *right direction*, MSE tells us how far off the predictions are *in actual points*. A system could rank essays perfectly but still be 2 points too high on everything — MSE catches that.

- MSE = 0 → every prediction exactly matches the human score
- Larger MSE → worse predictions

**Toy Example (same data):**

For each essay, compute the error and square it:

| Essay | Human | Predicted | Error | Squared Error |
|-------|-------|-----------|-------|---------------|
| A     | 2.0   | 2.5       | −0.5  | 0.25          |
| B     | 3.0   | 2.8       | 0.2   | 0.04          |
| C     | 4.0   | 3.5       | 0.5   | 0.25          |
| D     | 1.0   | 1.5       | −0.5  | 0.25          |
| E     | 5.0   | 4.7       | 0.3   | 0.09          |

Now just take the average of the squared errors:

$$
\text{MSE} = (0.25 + 0.04 + 0.25 + 0.25 + 0.09) \div 5 = 0.88 \div 5 = 0.176
$$

To get a feel in the original score units, take the square root: \(\sqrt{0.176} \approx 0.42\). So our predictions are off by roughly **0.4 points** on a 1–5 scale — quite good!

> **In Python:**
> ```python
> from sklearn.metrics import mean_squared_error
> mse = mean_squared_error([2.0, 3.0, 4.0, 1.0, 5.0],
>                          [2.5, 2.8, 3.5, 1.5, 4.7])
> print(f"MSE = {mse:.3f}")  # 0.176
> ```

---

**Why do we need both?**

Pearson r asks *"does the system know which essays are better?"* while MSE asks *"are the predicted scores close to the actual scores?"* A good AES system should have **high r** and **low MSE**.

---

### What is a "baseline"?

A **baseline** is a simple, naive method that any good model should **beat**. If your fancy model can't beat "always guess 3.0", something is wrong! We will build several baselines to understand the difficulty of the task.

In [ ]:
# === Evaluation function (used throughout the notebook) ===

def evaluate(y_true, y_pred, label='Model'):
    '''
    Compare predicted scores to true (human) scores.
    Returns Pearson r and Mean Squared Error (MSE).
    '''
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    # Mean Squared Error
    mse = mean_squared_error(y_true, y_pred)

    # Pearson correlation
    # Handle edge case: if all predictions are the same number,
    # correlation is undefined (no variance), so we set r = 0.
    if np.std(y_pred) < 1e-10:
        r = 0.0
        note = ' (constant prediction → r undefined, set to 0)'
    else:
        r, _ = pearsonr(y_true, y_pred)
        note = ''

    print(f'[{label}]')
    print(f'  Pearson r = {r:.4f}{note}')
    print(f'  MSE       = {mse:.4f}')
    print()
    return r, mse

print('evaluate() function defined ✅')

### Official train/test workflow

For the final submission, we follow the official project setup directly:
- use the full official `train` file to train feature-based models
- evaluate feature-based models on the full official `test` file
- run the LLM experiment on a stratified official-test subset

We keep a fixed random seed so all official evaluation steps remain reproducible.


In [ ]:
# Official evaluation settings
RANDOM_SEED = 42
FINAL_LLM_TEST_SIZE = 60

print(f'Official training set: {len(train_df)} essays')
print(f'Official test set:     {len(test_df)} essays')
print()
print(f'Final LLM evaluation subset size (official test): {FINAL_LLM_TEST_SIZE} essays')


### Baseline experiments

Let's see how well simple approaches perform on the **official test set** before we compare them to stronger regression and LLM-based systems.


In [ ]:
# Ground truth: the human Overall scores for the official test set
y_true_official = test_df['Overall'].values

# -------------------------------------------------------
# Baseline A1: Always predict the LOWEST score (1.0)
# -------------------------------------------------------
y_pred_a1 = np.ones(len(y_true_official)) * 1.0
r_a1, mse_a1 = evaluate(y_true_official, y_pred_a1, 'Baseline A1: Always 1.0')

# -------------------------------------------------------
# Baseline A2: Always predict the HIGHEST score (5.0)
# -------------------------------------------------------
y_pred_a2 = np.ones(len(y_true_official)) * 5.0
r_a2, mse_a2 = evaluate(y_true_official, y_pred_a2, 'Baseline A2: Always 5.0')

# -------------------------------------------------------
# Baseline A3: Always predict the MEAN score from training
# -------------------------------------------------------
train_mean = train_df['Overall'].mean()
y_pred_a3 = np.ones(len(y_true_official)) * train_mean
r_a3, mse_a3 = evaluate(y_true_official, y_pred_a3, f'Baseline A3: Always {train_mean:.2f} (train mean)')


In [ ]:
# -------------------------------------------------------
# Baseline B: Random scores between 1.0 and 5.0
# -------------------------------------------------------
np.random.seed(123)
y_pred_b = np.random.uniform(1.0, 5.0, size=len(y_true_official))
r_b, mse_b = evaluate(y_true_official, y_pred_b, 'Baseline B: Random (uniform 1-5)')


In [ ]:
# -------------------------------------------------------
# Baseline C: Use raw word count as the "score"
# -------------------------------------------------------
# This is clearly wrong (word counts are in the hundreds,
# but scores are 1-5), but it shows what happens when
# your predictions are on the wrong scale.
y_pred_c = test_df['num_words'].values.astype(float)
r_c, mse_c = evaluate(y_true_official, y_pred_c, 'Baseline C: Raw word count (wrong scale!)')

print('Notice: Pearson r might be OK (word count correlates with proficiency),')
print('but MSE is huge because the scale is completely wrong!')


In [ ]:
# -------------------------------------------------------
# Baseline D: Normalize word count to the 1-5 range
# -------------------------------------------------------
# Min-max normalization: rescale values from [min, max] to [1, 5]
# Formula: new = 1 + 4 * (old - min) / (max - min)

wc_min = train_df['num_words'].min()
wc_max = train_df['num_words'].max()

y_pred_d = 1.0 + 4.0 * (test_df['num_words'].values - wc_min) / (wc_max - wc_min)
y_pred_d = np.clip(y_pred_d, 1.0, 5.0)

r_d, mse_d = evaluate(y_true_official, y_pred_d, 'Baseline D: Normalized word count (1-5)')

print('Now the scale is correct, so MSE is much better.')
print('But can we do better with more features? -> Section 3!')


In [ ]:
# -------------------------------------------------------
# Summary table of all baselines
# -------------------------------------------------------
print('=' * 60)
print(f'{"Baseline":<42} {"Pearson r":>9} {"MSE":>8}')
print('=' * 60)
baselines = [
    ('A1: Always predict 1.0', r_a1, mse_a1),
    ('A2: Always predict 5.0', r_a2, mse_a2),
    (f'A3: Always predict {train_mean:.2f} (mean)', r_a3, mse_a3),
    ('B: Random (uniform 1-5)', r_b, mse_b),
    ('C: Raw word count (wrong scale)', r_c, mse_c),
    ('D: Normalized word count (1-5)', r_d, mse_d),
]
for name, r, mse in baselines:
    print(f'  {name:<40} {r:>9.4f} {mse:>8.4f}')
print('=' * 60)

### 🤔 Discussion: What did we learn from baselines?

1. **Constant predictions** (A1, A2, A3) have r = 0 because there is no variation — the model can't distinguish good from bad essays.
2. **Random predictions** (B) have r ≈ 0 and high MSE — guessing randomly is useless.
3. **Raw word count** (C) may show some correlation (longer essays tend to get higher scores), but the MSE is enormous because the *scale* is wrong.
4. **Normalized word count** (D) fixes the scale, so MSE improves. But using only one feature is limited.

**Takeaway:** Even a simple feature (word count) contains *some* information about proficiency. Can we do better with **multiple features**? → Yes! That's what linear regression does.

---

## Section 3: Task A - Linear Regression Baselines and Expanded Features

### What is Linear Regression?

Linear regression is a simple statistical model that predicts a score by combining several features with **weights**:

```text
predicted_score = w1 * feature1 + w2 * feature2 + ... + wn * featuren + b
```

Where:
- Each `w_i` is a **weight** (coefficient) that the model learns. It tells us how important each feature is.
- `b` is the **intercept** (bias).

### Two regression settings

We will compare two feature-based AES setups:

1. **LR-basic**: the original 5-feature notebook version
2. **LR-expanded**: a stronger feature set that adds engineered length and structure features

### LR-basic features

| Feature | What it measures |
|---------|------------------|
| `num_words` | Total number of words (essay length) |
| `num_sent` | Number of sentences |
| `TTR` | Type-Token Ratio (vocabulary diversity) |
| `MTLD` | Measure of Textual Lexical Diversity |
| `num_word_div_para` | Average words per paragraph |

### LR-expanded additions

The expanded model keeps the 5 basic features and adds:

- `num_para`
- `avg_sentence_length`
- `avg_paragraph_length`
- `sentences_per_paragraph`
- `log_num_words`
- `Type`
- `Token`

This gives us a more realistic feature-based AES comparison for the final report.


In [ ]:
# Build LR-basic and LR-expanded feature matrices

def add_engineered_features(frame):
    frame = frame.copy()
    frame['avg_sentence_length'] = frame['num_words'] / frame['num_sent'].clip(lower=1)
    frame['avg_paragraph_length'] = frame['num_words'] / frame['num_para'].clip(lower=1)
    frame['sentences_per_paragraph'] = frame['num_sent'] / frame['num_para'].clip(lower=1)
    frame['log_num_words'] = np.log1p(frame['num_words'])
    return frame

train_lr_df = add_engineered_features(train_df)
test_lr_df = add_engineered_features(test_df)

LR_BASIC_FEATURES = ['num_words', 'num_sent', 'TTR', 'MTLD', 'num_word_div_para']
LR_EXPANDED_FEATURES = LR_BASIC_FEATURES + [
    'num_para',
    'avg_sentence_length',
    'avg_paragraph_length',
    'sentences_per_paragraph',
    'log_num_words',
    'Type',
    'Token',
]

all_lr_features = LR_BASIC_FEATURES + [f for f in LR_EXPANDED_FEATURES if f not in LR_BASIC_FEATURES]
missing_feature_cols = [
    f for f in all_lr_features
    if f not in train_lr_df.columns or f not in test_lr_df.columns
]
if missing_feature_cols:
    raise ValueError(f'Missing feature columns for regression: {missing_feature_cols}')

X_train_basic = train_lr_df[LR_BASIC_FEATURES].values
y_train = train_lr_df['Overall'].values
X_test_basic = test_lr_df[LR_BASIC_FEATURES].values
y_test = test_lr_df['Overall'].values

X_train_expanded = train_lr_df[LR_EXPANDED_FEATURES].values
X_test_expanded = test_lr_df[LR_EXPANDED_FEATURES].values

print('LR-basic features:', LR_BASIC_FEATURES)
print('LR-expanded features:', LR_EXPANDED_FEATURES)
print()
print(f'Official train size: {len(train_lr_df)} essays')
print(f'Official test size:  {len(test_lr_df)} essays')
print(f'LR-basic matrix:     train {X_train_basic.shape}, test {X_test_basic.shape}')
print(f'LR-expanded matrix:  train {X_train_expanded.shape}, test {X_test_expanded.shape}')


In [ ]:
# Train both linear regression variants
lr_basic_model = LinearRegression()
lr_basic_model.fit(X_train_basic, y_train)

lr_expanded_model = LinearRegression()
lr_expanded_model.fit(X_train_expanded, y_train)

print('LR-basic model trained successfully.')
print('LR-expanded model trained successfully.')


In [ ]:
# Predict on the official test split and evaluate both models
y_pred_lr_basic = np.clip(lr_basic_model.predict(X_test_basic), 1.0, 5.0)
y_pred_lr_expanded = np.clip(lr_expanded_model.predict(X_test_expanded), 1.0, 5.0)

r_lr_basic, mse_lr_basic = evaluate(y_test, y_pred_lr_basic, 'LR-basic (5 features)')
r_lr_expanded, mse_lr_expanded = evaluate(y_test, y_pred_lr_expanded, 'LR-expanded (12 features)')

lr_candidates = [
    {
        'name': 'LR-basic',
        'model': lr_basic_model,
        'features': LR_BASIC_FEATURES,
        'predictions': y_pred_lr_basic,
        'pearson_r': r_lr_basic,
        'mse': mse_lr_basic,
    },
    {
        'name': 'LR-expanded',
        'model': lr_expanded_model,
        'features': LR_EXPANDED_FEATURES,
        'predictions': y_pred_lr_expanded,
        'pearson_r': r_lr_expanded,
        'mse': mse_lr_expanded,
    },
]

selected_lr = max(lr_candidates, key=lambda item: (item['pearson_r'], -item['mse']))
selected_lr_name = selected_lr['name']
lr_model = selected_lr['model']
FEATURES = selected_lr['features']
y_pred_lr = selected_lr['predictions']
r_lr = selected_lr['pearson_r']
mse_lr = selected_lr['mse']

print(f'Selected default export model: {selected_lr_name}')
print(f'Selected feature count:        {len(FEATURES)}')


In [ ]:
# Inspect feature weights for both linear regression variants

def print_coefficient_table(label, feature_names, model):
    coef_df = pd.DataFrame({
        'Feature': feature_names,
        'Weight': model.coef_,
    })
    coef_df['AbsWeight'] = coef_df['Weight'].abs()
    coef_df = coef_df.sort_values('AbsWeight', ascending=False).drop(columns='AbsWeight')

    print(label)
    print('-' * 60)
    print(coef_df.to_string(index=False))
    print(f'Intercept: {model.intercept_:.6f}')
    print()

print_coefficient_table('LR-basic coefficients', LR_BASIC_FEATURES, lr_basic_model)
print_coefficient_table('LR-expanded coefficients', LR_EXPANDED_FEATURES, lr_expanded_model)
print(f'Default export model for later cells: {selected_lr_name}')


In [ ]:
# Scatter plots: LR-basic vs LR-expanded on the official test split
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
plot_specs = [
    ('LR-basic', y_pred_lr_basic, r_lr_basic, 'steelblue'),
    ('LR-expanded', y_pred_lr_expanded, r_lr_expanded, 'darkorange'),
]

for ax, (label, preds, corr, color) in zip(axes, plot_specs):
    ax.scatter(y_test, preds, alpha=0.45, edgecolors='black', linewidths=0.4, color=color)
    ax.plot([1, 5], [1, 5], 'r--', linewidth=1)
    ax.set_title(f'{label} (r = {corr:.3f})', fontsize=12)
    ax.set_xlabel('Human Score')
    ax.set_ylabel('Predicted Score')
    ax.set_xlim(0.5, 5.5)
    ax.set_ylim(0.5, 5.5)
    ax.grid(True, alpha=0.3)

plt.suptitle('Linear Regression Comparison on the Official Test Split', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Updated comparison table: baselines + both linear regression variants
print('=' * 68)
print(f'{"Model":<46} {"Pearson r":>9} {"MSE":>8}')
print('=' * 68)
all_results = baselines + [
    ('LR-basic (5 features)', r_lr_basic, mse_lr_basic),
    ('LR-expanded (12 features)', r_lr_expanded, mse_lr_expanded),
]
for name, r, mse in all_results:
    print(f'  {name:<44} {r:>9.4f} {mse:>8.4f}')
print('=' * 68)
print()
print(f'Default export model for later cells: {selected_lr_name}')
print('Next, we can move on to the official-test LLM scoring section.')


---
## Section 4: Task B - LLM Scoring with OpenRouter API

### What is an LLM API?

Instead of extracting features and training a model, we can ask a **Large Language Model** to read an essay and directly output a score. This is a generative AI approach to AES.

### Why OpenRouter here?

For this project, we use **OpenRouter's OpenAI-compatible API** and compare:

- 3 models
- 2 prompt versions
- 2 temperatures

That gives us **12 LLM conditions**.

### What this notebook does in Section 4

For the final submission, we evaluate all 12 LLM conditions on a **stratified 60-essay subset** sampled from the official `test` file.


In [ ]:
# OpenRouter API client setup
from openai import OpenAI
import os

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass('Enter your OpenRouter API key: ')

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
)

print('OpenRouter API client ready.')
print('Base URL: https://openrouter.ai/api/v1')


### Rubric and Prompt Setup

This notebook now reads the official local file `ELL_Rubrics.docx` directly from the project folder and uses its **holistic overall rubric** inside the scoring prompt.

If this cell cannot read the docx file, stop and fix the rubric path or parsing issue before running any LLM scoring experiments.


In [ ]:
# Prompt library for the 12-condition OpenRouter experiment
from zipfile import ZipFile
from xml.etree import ElementTree as ET

ALLOWED_SCORES = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
RUBRIC_DOCX_PATH = Path('ELL_Rubrics.docx')

def extract_docx_table_rows(docx_path):
    with ZipFile(docx_path) as zf:
        xml_bytes = zf.read('word/document.xml')
    root = ET.fromstring(xml_bytes)
    ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}

    def cell_text(tc):
        texts = [t.text for t in tc.findall('.//w:t', ns) if t.text]
        return ' '.join(''.join(texts).split())

    tables = []
    for tbl in root.findall('.//w:tbl', ns):
        rows = []
        for tr in tbl.findall('./w:tr', ns):
            cells = [cell_text(tc) for tc in tr.findall('./w:tc', ns)]
            if any(cells):
                rows.append(cells)
        if rows:
            tables.append(rows)
    return tables


def build_holistic_rubric_from_docx(docx_path):
    tables = extract_docx_table_rows(docx_path)
    if not tables:
        raise ValueError('Could not locate any table structure in ELL_Rubrics.docx')

    rubric_table = tables[0]
    if len(rubric_table) < 3:
        raise ValueError('The rubric table in ELL_Rubrics.docx is missing expected rows')

    header_row = rubric_table[1]
    if 'Overall' not in header_row:
        raise ValueError('Could not locate the Overall column in ELL_Rubrics.docx')

    overall_col_idx = header_row.index('Overall')
    score_map = {}
    for row in rubric_table[2:]:
        if len(row) <= overall_col_idx:
            continue
        score = row[0].strip()
        overall_text = row[overall_col_idx].strip()
        if score in {'5', '4', '3', '2', '1'} and overall_text:
            score_map[score] = overall_text

    if not score_map:
        raise ValueError('Could not extract holistic score descriptions from the Overall column in ELL_Rubrics.docx')

    ordered_scores = ['5', '4', '3', '2', '1']
    lines = ['Official holistic rubric extracted from ELL_Rubrics.docx:']
    for score in ordered_scores:
        if score in score_map:
            lines.append(f"{score}.0 - {score_map[score]}")
    lines.append('Use 4.5, 3.5, 2.5, and 1.5 for essays that fall between adjacent score bands.')
    return '\n'.join(lines)


if not RUBRIC_DOCX_PATH.exists():
    raise FileNotFoundError(f'Could not find the rubric file: {RUBRIC_DOCX_PATH.resolve()}')

RUBRIC_TEXT = build_holistic_rubric_from_docx(RUBRIC_DOCX_PATH)
RUBRIC_SOURCE = str(RUBRIC_DOCX_PATH)
print(f'Loaded official rubric text from: {RUBRIC_SOURCE}')

PROMPT_LIBRARY = {
    'P1_direct_holistic': f'''You are an expert assessor of English Language Learner (ELL) writing.

Your job is to assign one holistic proficiency score to the student's essay using the rubric below.

{RUBRIC_TEXT}

Instructions:
- Score only the quality of the student's English writing.
- Focus on language proficiency, clarity, grammar, syntax, vocabulary, cohesion, and organization.
- Do not judge the essay based on whether you agree with the student's opinion.
- Do not judge the essay based on factual correctness unless it directly affects comprehensibility.
- Use only these allowed scores: {ALLOWED_SCORES}
- Respond with exactly one score and no explanation.''',

    'P2_analytic_to_holistic': f'''You are an expert assessor of English Language Learner (ELL) writing.

Assign one holistic proficiency score to the student's essay using the rubric below.

{RUBRIC_TEXT}

Before choosing the final score, internally consider:
- comprehensibility
- organization and cohesion
- grammar and syntax control
- vocabulary range and word choice
- sentence variety and fluency

Then map your internal judgment to one holistic score.

Instructions:
- Score only the quality of the student's English writing.
- Do not judge the essay based on opinion quality or topic stance.
- Use only these allowed scores: {ALLOWED_SCORES}
- Respond with exactly one score and no explanation.''',
}

print('Rubric source:', RUBRIC_SOURCE)
print('Prompt versions ready:', list(PROMPT_LIBRARY.keys()))
for prompt_name, prompt_text in PROMPT_LIBRARY.items():
    print(f'  {prompt_name}: {len(prompt_text)} characters')


In [ ]:
# OpenRouter model registry for the final 12-condition comparison
MODEL_REGISTRY = {
    'deepseek_v3_2': 'deepseek/deepseek-v3.2',
    'mistral_small_3_1_24b': 'mistralai/mistral-small-3.1-24b-instruct',
    'gemma_3_27b_it': 'google/gemma-3-27b-it',
}

print('Registered OpenRouter models:')
for short_name, model_id in MODEL_REGISTRY.items():
    print(f'  {short_name:<24} -> {model_id}')


In [ ]:
# Experiment settings: 3 models x 2 prompts x 2 temperatures = 12 conditions
TEMPERATURES = [0.0, 0.2]
MAX_TOKENS = 8
TOP_P = 1.0
REQUEST_DELAY_SECONDS = 0.4
MAX_RETRIES = 2
LLM_FINAL_SIZE = FINAL_LLM_TEST_SIZE

_, llm_eval_df = train_test_split(
    test_df,
    test_size=LLM_FINAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=test_df['Overall'],
)
llm_eval_df = llm_eval_df.reset_index(drop=True).copy()

EXPERIMENT_CONFIGS = []
for model_key, model_id in MODEL_REGISTRY.items():
    for prompt_key, prompt_text in PROMPT_LIBRARY.items():
        for temperature in TEMPERATURES:
            temp_tag = str(temperature).replace('.', 'p')
            experiment_id = f'{model_key}__{prompt_key}__t{temp_tag}'
            EXPERIMENT_CONFIGS.append({
                'experiment_id': experiment_id,
                'model_key': model_key,
                'model_id': model_id,
                'prompt_key': prompt_key,
                'system_prompt': prompt_text,
                'temperature': temperature,
            })

print(f'Official test subset size for final LLM evaluation: {len(llm_eval_df)} essays')
print('Score distribution in llm_eval_df (official test subset):')
print(llm_eval_df['Overall'].value_counts().sort_index())
print()
print(f'Total experiment conditions: {len(EXPERIMENT_CONFIGS)}')
for config in EXPERIMENT_CONFIGS:
    print(f"  {config['experiment_id']}")


In [ ]:
# Core OpenRouter scoring and parsing helpers

def normalize_to_allowed_score(value, allowed_scores):
    value = max(min(value, max(allowed_scores)), min(allowed_scores))
    return min(allowed_scores, key=lambda x: abs(x - value))


def parse_llm_score(reply, allowed_scores=ALLOWED_SCORES):
    if reply is None:
        return None

    numbers = re.findall(r'-?\d+(?:\.\d+)?', str(reply))
    if not numbers:
        return None

    raw_value = float(numbers[0])
    return float(normalize_to_allowed_score(raw_value, allowed_scores))


def score_essay_with_llm(essay_text, client, model, system_prompt, temperature, max_tokens, top_p=1.0):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': f'Score this essay. Return only one score.\n\nEssay:\n{essay_text}'},
            ],
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
            stream=False,
        )

        reply = response.choices[0].message.content
        if isinstance(reply, list):
            reply = ' '.join(str(part) for part in reply)
        reply = str(reply).strip()

        score = parse_llm_score(reply)
        return score, reply

    except Exception as e:
        return None, f'ERROR: {str(e)}'

print('score_essay_with_llm() helper ready.')


In [ ]:
# Optional sanity check: score 2 essays with the first OpenRouter configuration
sample_config = EXPERIMENT_CONFIGS[0]
print('Sanity check configuration:')
print(sample_config)
print('\nScoring 2 essays from the official test subset...')

for i in range(min(2, len(llm_eval_df))):
    essay = str(llm_eval_df[TEXT_COL].iloc[i])
    human_score = llm_eval_df['Overall'].iloc[i]

    llm_score, raw_reply = score_essay_with_llm(
        essay_text=essay,
        client=client,
        model=sample_config['model_id'],
        system_prompt=sample_config['system_prompt'],
        temperature=sample_config['temperature'],
        max_tokens=MAX_TOKENS,
        top_p=TOP_P,
    )

    print()
    print(f'Essay {i + 1}')
    print(f'  Human score:   {human_score}')
    print(f'  LLM score:     {llm_score}')
    print(f'  Raw reply:     {raw_reply}')
    time.sleep(REQUEST_DELAY_SECONDS)

print('\nIf the replies look reasonable, continue to the full 12-condition run.')


In [ ]:
# Sequential runner for one experiment configuration

def run_llm_experiment(eval_df, client, config, max_tokens, top_p, request_delay_seconds, max_retries):
    scores = []
    raw_replies = []
    failed_local_indices = []

    for local_idx, row in eval_df.reset_index(drop=True).iterrows():
        essay_text = str(row[TEXT_COL])
        score = None
        raw_reply = None

        for attempt in range(max_retries + 1):
            score, raw_reply = score_essay_with_llm(
                essay_text=essay_text,
                client=client,
                model=config['model_id'],
                system_prompt=config['system_prompt'],
                temperature=config['temperature'],
                max_tokens=max_tokens,
                top_p=top_p,
            )
            if score is not None:
                break
            time.sleep(request_delay_seconds * (attempt + 1))

        if score is None:
            failed_local_indices.append(local_idx)

        scores.append(score)
        raw_replies.append(raw_reply)
        time.sleep(request_delay_seconds)

    valid_mask = np.array([score is not None for score in scores])
    y_true = eval_df['Overall'].values[valid_mask]
    y_pred = np.array([score for score in scores if score is not None], dtype=float)

    if len(y_pred) >= 2:
        r_value, mse_value = evaluate(
            y_true,
            y_pred,
            label=f"{config['model_key']} | {config['prompt_key']} | t={config['temperature']}"
        )
    else:
        r_value, mse_value = np.nan, np.nan
        print(f"[{config['experiment_id']}] Not enough valid predictions to compute metrics.\n")

    return {
        'essay_ids': eval_df[ESSAY_ID_COL].tolist(),
        'scores': scores,
        'raw_replies': raw_replies,
        'failed_indices': failed_local_indices,
        'successful_count': int(valid_mask.sum()),
        'pearson_r': None if pd.isna(r_value) else float(r_value),
        'mse': None if pd.isna(mse_value) else float(mse_value),
    }

print('run_llm_experiment() helper ready.')


In [ ]:
# Run all 12 OpenRouter LLM experiments on the official-test subset
llm_experiment_results = {}
llm_result_rows = []

for run_idx, config in enumerate(EXPERIMENT_CONFIGS, start=1):
    print('=' * 88)
    print(f"Running experiment {run_idx}/{len(EXPERIMENT_CONFIGS)}: {config['experiment_id']}")
    print(f"Model: {config['model_id']}")
    print(f"Prompt: {config['prompt_key']} | Temperature: {config['temperature']}")
    print('=' * 88)

    experiment_result = run_llm_experiment(
        eval_df=llm_eval_df,
        client=client,
        config=config,
        max_tokens=MAX_TOKENS,
        top_p=TOP_P,
        request_delay_seconds=REQUEST_DELAY_SECONDS,
        max_retries=MAX_RETRIES,
    )
    llm_experiment_results[config['experiment_id']] = {
        'config': config,
        **experiment_result,
    }

    llm_result_rows.append({
        'experiment_id': config['experiment_id'],
        'model_key': config['model_key'],
        'model_id': config['model_id'],
        'prompt_key': config['prompt_key'],
        'temperature': config['temperature'],
        'successful_count': experiment_result['successful_count'],
        'num_failed': len(experiment_result['failed_indices']),
        'pearson_r': experiment_result['pearson_r'],
        'mse': experiment_result['mse'],
    })

llm_results_df = pd.DataFrame(llm_result_rows)
llm_results_df = llm_results_df.sort_values(['pearson_r', 'mse'], ascending=[False, True], na_position='last').reset_index(drop=True)
print('\nOfficial-test LLM runs completed.')


### Official LLM Results

The cells below summarize the **12 OpenRouter LLM conditions** on the official 60-essay test subset and select the best-performing final configuration for export.


In [ ]:
# Summarize the 12 LLM conditions and select the best one for later export
summary_cols = ['experiment_id', 'model_key', 'prompt_key', 'temperature', 'successful_count', 'num_failed', 'pearson_r', 'mse']
print('LLM official-test summary (sorted by Pearson r, then MSE):')
print(llm_results_df[summary_cols].to_string(index=False))

best_llm_row = llm_results_df.iloc[0]
best_experiment_id = best_llm_row['experiment_id']
best_llm_result = llm_experiment_results[best_experiment_id]

MODEL_NAME = best_llm_result['config']['model_id']
TEMPERATURE = best_llm_result['config']['temperature']
SYSTEM_PROMPT = best_llm_result['config']['system_prompt']
PROMPT_VERSION = best_llm_result['config']['prompt_key']
llm_scores = best_llm_result['scores']
llm_raw_replies = best_llm_result['raw_replies']
failed_indices = best_llm_result['failed_indices']

y_true_llm = np.array(llm_eval_df['Overall'].values, dtype=float)
y_pred_llm = np.array(llm_scores, dtype=float)
r_llm = best_llm_result['pearson_r']
mse_llm = best_llm_result['mse']

print()
print('Selected best LLM configuration for final export:')
print(f'  Experiment ID: {best_experiment_id}')
print(f'  Model:         {MODEL_NAME}')
print(f'  Prompt:        {PROMPT_VERSION}')
print(f'  Temperature:   {TEMPERATURE}')
print(f'  Pearson r:     {r_llm:.4f}')
print(f'  MSE:           {mse_llm:.4f}')


In [ ]:
# Scatter plot: best OpenRouter LLM configuration vs. human scores
plt.figure(figsize=(6.5, 5))
plt.scatter(y_true_llm, y_pred_llm, alpha=0.55, edgecolors='black', linewidths=0.4, color='darkorange')
plt.plot([1, 5], [1, 5], 'r--', linewidth=1)
plt.xlabel('Human Score')
plt.ylabel('Predicted Score')
plt.title(f'Best OpenRouter LLM vs. Human Scores (Official Test Subset, r = {r_llm:.3f})', fontsize=13)
plt.xlim(0.5, 5.5)
plt.ylim(0.5, 5.5)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Final comparison table: baselines, regression models, and all 12 OpenRouter LLM runs
comparison_rows = []
for name, r_value, mse_value in baselines:
    comparison_rows.append({
        'method': name,
        'category': 'Baseline',
        'evaluation_scope': 'official_test',
        'Pearson r': r_value,
        'MSE': mse_value,
    })

comparison_rows.extend([
    {
        'method': 'LR-basic (5 features)',
        'category': 'Linear regression',
        'evaluation_scope': 'official_test',
        'Pearson r': r_lr_basic,
        'MSE': mse_lr_basic,
    },
    {
        'method': 'LR-expanded (12 features)',
        'category': 'Linear regression',
        'evaluation_scope': 'official_test',
        'Pearson r': r_lr_expanded,
        'MSE': mse_lr_expanded,
    },
])

for _, row in llm_results_df.iterrows():
    comparison_rows.append({
        'method': f"LLM | {row['model_key']} | {row['prompt_key']} | t={row['temperature']}",
        'category': 'LLM',
        'evaluation_scope': f'official_test_subset_{len(llm_eval_df)}',
        'Pearson r': row['pearson_r'],
        'MSE': row['mse'],
    })

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))


In [ ]:
# Visualization: baselines + both LR models + best OpenRouter LLM
model_names = [
    'Always 1.0',
    'Always 5.0',
    f'Mean ({train_mean:.1f})',
    'Random',
    'Norm. Length',
    'LR-basic',
    'LR-expanded',
    'Best OpenRouter LLM',
]
r_values = [r_a1, r_a2, r_a3, r_b, r_d, r_lr_basic, r_lr_expanded, r_llm]
colors = ['#d9d9d9', '#d9d9d9', '#d9d9d9', '#d9d9d9', '#a6cee3', '#1f78b4', '#33a02c', '#ff7f00']

plt.figure(figsize=(11, 5))
bars = plt.bar(model_names, r_values, color=colors, edgecolor='black', linewidth=0.5)
plt.ylabel('Pearson r', fontsize=12)
plt.title('Comparison of Scoring Methods on Official Evaluation Data', fontsize=13)
plt.xticks(rotation=25, ha='right', fontsize=10)
plt.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
plt.ylim(-0.1, max(r_values) + 0.15)

for bar, value in zip(bars, r_values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f'{value:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


### Reflection Questions

1. Which of the 12 OpenRouter LLM conditions performed best on the official test subset?
2. Did the best LLM beat `LR-expanded` on Pearson `r`, on `MSE`, or both?
3. Did prompt version matter more than temperature?
4. Did model choice matter more than prompt wording?
5. What are the main strengths and limitations of feature-based AES versus LLM-based AES in this official evaluation?

---


## Section 5: Save Results for Submission

Run the cells below to save your **official evaluation** results. You will need to submit:
1. **part1_predictions.csv** - official-test human scores, LR predictions, and LLM predictions
2. **part1_config.json** - the final official evaluation configuration

This final notebook now saves results for the **official test workflow**.


In [ ]:
# Save predictions to CSV
results_df = test_df[[ESSAY_ID_COL, 'Overall']].copy()
results_df = results_df.rename(columns={ESSAY_ID_COL: 'essay_id', 'Overall': 'human_score'})
results_df['evaluation_split'] = 'official_test'
results_df['llm_final_selected'] = results_df['essay_id'].isin(set(llm_eval_df[ESSAY_ID_COL])).astype(int)
results_df['lr_basic_prediction'] = y_pred_lr_basic
results_df['lr_expanded_prediction'] = y_pred_lr_expanded
results_df['lr_prediction'] = y_pred_lr

for experiment_id, experiment_payload in llm_experiment_results.items():
    score_map = dict(zip(experiment_payload['essay_ids'], experiment_payload['scores']))
    results_df[f'{experiment_id}__score'] = results_df['essay_id'].map(score_map)

best_score_map = dict(zip(best_llm_result['essay_ids'], llm_scores))
best_reply_map = dict(zip(best_llm_result['essay_ids'], llm_raw_replies))
results_df['llm_prediction'] = results_df['essay_id'].map(best_score_map)
results_df['llm_raw_reply'] = results_df['essay_id'].map(best_reply_map)

results_df.to_csv('part1_predictions.csv', index=False)
print('Saved predictions to part1_predictions.csv')
print(f'   {len(results_df)} rows saved.')
print()
results_df.head(10)


In [ ]:
# Save model configuration to JSON
config = {
    'project': 'CALL Part 1: Automated Essay Scoring',
    'data': {
        'train_source': TRAIN_SOURCE,
        'test_source': TEST_SOURCE,
        'official_train_size': len(train_df),
        'official_test_size': len(test_df),
        'random_seed': RANDOM_SEED,
        'essay_id_column': ESSAY_ID_COL,
        'text_column': TEXT_COL,
    },
    'linear_regression': {
        'selected_variant': selected_lr_name,
        'selected_features': FEATURES,
        'selected_coefficients': lr_model.coef_.tolist(),
        'selected_intercept': float(lr_model.intercept_),
        'selected_pearson_r': round(float(r_lr), 4),
        'selected_mse': round(float(mse_lr), 4),
        'evaluation_split': 'official_test',
        'variants': {
            'lr_basic': {
                'features': LR_BASIC_FEATURES,
                'coefficients': lr_basic_model.coef_.tolist(),
                'intercept': float(lr_basic_model.intercept_),
                'pearson_r': round(float(r_lr_basic), 4),
                'mse': round(float(mse_lr_basic), 4),
            },
            'lr_expanded': {
                'features': LR_EXPANDED_FEATURES,
                'coefficients': lr_expanded_model.coef_.tolist(),
                'intercept': float(lr_expanded_model.intercept_),
                'pearson_r': round(float(r_lr_expanded), 4),
                'mse': round(float(mse_lr_expanded), 4),
            },
        },
    },
    'llm': {
        'provider': 'OpenRouter',
        'api_base_url': 'https://openrouter.ai/api/v1',
        'evaluation_split': 'official_test_subset',
        'evaluation_subset_size': len(llm_eval_df),
        'evaluation_essay_ids': llm_eval_df[ESSAY_ID_COL].tolist(),
        'registered_models': MODEL_REGISTRY,
        'prompt_versions': list(PROMPT_LIBRARY.keys()),
        'temperatures': TEMPERATURES,
        'max_tokens': MAX_TOKENS,
        'top_p': TOP_P,
        'request_delay_seconds': REQUEST_DELAY_SECONDS,
        'max_retries': MAX_RETRIES,
        'selected_experiment_id': best_experiment_id,
        'selected_model_name': MODEL_NAME,
        'selected_prompt_version': PROMPT_VERSION,
        'selected_temperature': TEMPERATURE,
        'system_prompt': SYSTEM_PROMPT,
        'pearson_r': round(float(r_llm), 4),
        'mse': round(float(mse_llm), 4),
        'num_successfully_scored': int(len(y_pred_llm)),
        'num_failed': len(failed_indices),
        'experiments': llm_results_df.to_dict(orient='records'),
    },
}

with open('part1_config.json', 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print('Saved configuration to part1_config.json')
print()

# Display the config
print(json.dumps(config, indent=2, ensure_ascii=False))


In [ ]:
# Download the files to your computer
try:
    from google.colab import files
    files.download('part1_predictions.csv')
    files.download('part1_config.json')
    print('Files downloaded!')
except ImportError:
    print('Not running in Google Colab.')
    print('Please manually download the files from the file browser on the left.')

---
## Submission Checklist

Before submitting, make sure you have:

- [ ] Used the official rubric from `ELL_Rubrics.docx`
- [ ] Compared 3 models x 2 prompts x 2 temperatures = 12 LLM conditions
- [ ] Evaluated the final LLM experiment on a stratified official-test subset (`60` essays)
- [ ] Trained and evaluated the linear regression models with the official train/test workflow
- [ ] Run all cells and checked the final comparison table
- [ ] Saved **part1_predictions.csv**
- [ ] Saved **part1_config.json**
- [ ] Saved this notebook (`.ipynb`) with final outputs visible
- [ ] Exported the notebook to PDF for submission

### Files to submit for Part 1
1. PDF export of this notebook
2. `part1_predictions.csv`
3. `part1_config.json`

### In your report, discuss
- What prompt design choices did you make and why?
- Which model, prompt, and temperature performed best?
- How does the LLM compare to linear regression?
- Which features were most predictive in the regression model?
- What are the strengths and weaknesses of each approach?


In [ ]:
# NOTE: This cell is for Google Colab export only. Skip running it in VS Code.

### Convert to html
from google.colab import _message
import json

# Fetch the current notebook's content
notebook = _message.blocking_request('get_ipynb')

# Write it to a file
with open('/content/mynb.ipynb', 'w') as f:
    json.dump(notebook['ipynb'], f)

# Convert
!jupyter nbconvert --to html /content/mynb.ipynb

In [ ]:
# NOTE: This cell is for Google Colab / Linux PDF export only. Skip running it in VS Code.

# Page setup
!printf '@page {\nsize: A4;\nmargin: 1cm;\n}\n' > /content/custom.css

# Convert to pdf
!pip install weasyprint
!apt-get install -y fonts-noto-color-emoji
!weasyprint /content/mynb.html /content/mynb.pdf  -s /content/custom.css

---
*End of Part 1 Notebook*